In [1]:
import torch
import random 
import torch.nn.functional as F

In [10]:
#declare all variables
block_size=3
neurons=200
dimension=10
max_steps=200000

In [4]:
words=open("names.txt").read().splitlines()
stoi={chr(ord('a')+i):i+1 for i in range(26)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
def build_dataset(words):
    x,y=[],[]
    for w in words:
        context=[0]*block_size
        for ch in w+'.':
            ix=stoi[ch]
            x.append(context)
            y.append(ix)
            context=context[1:]+[ix]
    x=torch.tensor(x)
    y=torch.tensor(y)
    return x,y


In [11]:
#data-split
random.shuffle(words)
n1=int(0.8*len(words))
n2=int(0.9*len(words))
Xtr,Ytr=build_dataset(words[:n1])
Xval,Yval=build_dataset(words[n1:n2])
Xtest,Ytest=build_dataset(words[n2:])

In [12]:
#initialize all weights and embeddings
C=torch.randn((27,dimension))
W1=torch.randn((block_size*dimension,neurons))
b1=torch.randn(neurons)
W2=torch.randn((neurons,27))
b2=torch.randn(27)
parameters=[C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad=True


In [16]:
for i in range(max_steps):
    #mini-batch construct
    ix=torch.randint(0,Xtr.shape[0],(32,))
    emb=C[Xtr[ix]]
    #forward-pass
    h=torch.tanh(emb.view(-1,dimension*block_size)@W1+b1)
    logits=h@W2+b2
    loss=F.cross_entropy(logits,Ytr[ix])
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    lr=0.1 if i<max_steps/2 else 0.01
    for p in parameters:
        p.data+=-lr*p.grad
    